# PredictED: prediction and explainability
The three checkpoints and their applicability-domain artifacts are loaded once.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd() if (Path.cwd() / 'data').is_dir() else Path.cwd().parent
sys.path.insert(0, str(root))
from train.predictor import load_runtime, predict

runtime = load_runtime()
smiles = 'CC1=C(C=CC(=C1)OP(=S)(OC)OC)[N+](=O)[O-]'
result = predict(smiles, runtime)
result['hershberger'], result['uterotrophic'], result['tier1'][:3]

## Tier-2 explainability

In [ ]:
from train.tier2_loader import graph_loader_inference
from train.predictor import predict_tier2
from explanation.reveal import view_attention, view_counterfactuality
from IPython.display import SVG, display

mol_loader = graph_loader_inference(smiles, runtime)
for name in ('hershberger', 'uterotrophic'):
    reference_probability = predict_tier2(runtime.models[name], mol_loader, runtime.device)[0][0, 0]
    assert abs(float(reference_probability) - result[name]['probability']) < 1e-5

In [ ]:
for name in ('hershberger', 'uterotrophic'):
    attention_dir = root / 'output' / 'explanations' / name / 'attention'
    view_attention(runtime.models[name], mol_loader, runtime.device, attention_dir, pathway=name)
    display(SVG(filename=str(attention_dir / 'sample_1.svg')))

In [ ]:
import random
import numpy as np
import torch

for name in ('hershberger', 'uterotrophic'):
    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)
    counterfactual_dir = root / 'output' / 'explanations' / name / 'counterfactual'
    view_counterfactuality(
        runtime.models[name], mol_loader, root / 'data' / 'dataset.csv',
        root / 'data' / 'info.csv', runtime.device, counterfactual_dir,
        perturbations=15, task_index=0
    )
    display(SVG(filename=str(counterfactual_dir / runtime.assays[0] / 'sample_1.svg')))